# Mixture of Experts: The Experts

Once the router selects the experts, we need to process the tokens using those experts. Each expert is typically a simple FeedForward Network.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Expert(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff, bias=False),
            nn.SiLU(), # We use SiLU activation
            nn.Linear(d_ff, d_model, bias=False)
        )

    def forward(self, x):
        return self.net(x)

## Putting it together

The `MixtureOfExperts` layer combines the router and the experts. It routes tokens to the correct experts and sums their outputs weighted by the router's confidence.

In [ ]:
# Simplified logic (conceptual)
# In practice, we use efficient gathering and scattering operations to avoid loops.

def forward_moe(x, router, experts):
    weights, indices = router(x)
    output = torch.zeros_like(x)
    
    # For each token in the batch/sequence
    for b in range(x.size(0)):
        for t in range(x.size(1)):
            # For each selected expert (top-k)
            for k in range(indices.size(-1)):
                expert_idx = indices[b, t, k]
                weight = weights[b, t, k]
                
                # Run the expert
                expert_out = experts[expert_idx](x[b, t].unsqueeze(0))
                
                # Add weighted output
                output[b, t] += weight * expert_out.squeeze(0)
                
    return output